# Data Cleaning — Dirty Cafe Sales Dataset

**Objective:** Take a deliberately messy cafe point-of-sale export and systematically transform it into a clean, analysis-ready dataset, documenting every decision along the way.

**Dataset:** `dirty_cafe_sales.csv` — 10,000 transaction rows with 8 columns. The file was read entirely as text (`dtype=object`) because every column contains literal placeholder strings (`"ERROR"`, `"UNKNOWN"`) mixed in with genuine blanks and valid values.

---

## 1. Setup & Initial Load

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df_raw = pd.read_csv('dirty_cafe_sales.csv')
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(10)

Shape: 10000 rows x 8 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


## 2. Data Quality Report — Before Cleaning

Before touching anything, we characterise exactly how messy the data is: null counts, dtype issues, duplicate rows, and value-range anomalies. This becomes the "before" half of the before/after comparison in Section 8.

In [2]:
print("Column dtypes (as loaded):")
print(df_raw.dtypes)
print()
print("Literal null (NaN) count per column:")
print(df_raw.isnull().sum())

Column dtypes (as loaded):
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

Literal null (NaN) count per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


**Observation:** every column loaded as `object` (text), even the numeric ones (`Quantity`, `Price Per Unit`, `Total Spent`) and the date column — a strong signal that non-numeric placeholder text is mixed into otherwise numeric/date fields. The raw null counts above only capture truly blank cells; the next check reveals a second, hidden category of "missing" data.

In [3]:
# Value-range anomaly check: literal placeholder strings hiding as valid-looking values
placeholder_counts = {}
for col in df_raw.columns:
    vc = df_raw[col].value_counts(dropna=False)
    placeholder_counts[col] = {
        'ERROR': vc.get('ERROR', 0),
        'UNKNOWN': vc.get('UNKNOWN', 0),
        'true_NaN': df_raw[col].isnull().sum()
    }
placeholder_df = pd.DataFrame(placeholder_counts).T
placeholder_df['Total combined missing'] = placeholder_df.sum(axis=1)
placeholder_df

,ERROR,UNKNOWN,true_NaN,Total combined missing
Transaction ID,0,0,0,0
Item,292,344,333,969
Quantity,170,171,138,479
Price Per Unit,190,164,179,533
Total Spent,164,165,173,502
Payment Method,306,293,2579,3178
Location,358,338,3265,3961
Transaction Date,142,159,159,460


**Observation — the real missing-data picture:** every column hides two additional "missing" markers, the literal strings `"ERROR"` and `"UNKNOWN"`, on top of true blank cells. `Payment Method` (~32%) and `Location` (~40%) are the worst affected. Treating only true `NaN` as missing would have massively understated how incomplete this dataset actually is — so the very first cleaning step must be to unify all three representations into a single missing-value marker.

In [4]:
# Duplicate check (before any cleaning)
print(f"Fully duplicate rows: {df_raw.duplicated().sum()}")
print(f"Duplicate Transaction IDs: {df_raw['Transaction ID'].duplicated().sum()}")

Fully duplicate rows: 0
Duplicate Transaction IDs: 0


**Observation:** no fully duplicate rows and no duplicate `Transaction ID` values exist in the raw file — this dataset's messiness is entirely about missing/placeholder values and dtype issues, not repeated records. We still record this explicitly rather than assuming it.

## 3. Standardisation

Two standardisation issues need fixing before any imputation can happen:

1. **Unify missing-value representations.** `"ERROR"` and `"UNKNOWN"` are replaced with proper `NaN` across every column, so pandas' own null-handling tools (`isnull()`, `fillna()`, etc.) work correctly instead of treating these as valid category values.
2. **Correct data types.** `Transaction Date` is converted to real `datetime64`; `Quantity`, `Price Per Unit`, and `Total Spent` are converted to numeric. `Transaction ID`, `Item`, `Payment Method`, and `Location` remain strings/categoricals (an ID should never be treated as a number to be averaged).

Categorical text values (`"Credit Card"`, `"Cash"`, `"Digital Wallet"`, `"Takeaway"`, `"In-store"`) were checked and are already consistently capitalised in this dataset — there is no `"cash"` / `"CASH"` / `"Cash "` style inconsistency to fix here, unlike a typical gender or country field.

In [5]:
df = df_raw.copy()

# 1. Unify placeholder markers into real NaN
df = df.replace(['ERROR', 'UNKNOWN'], np.nan)

# 2. Correct dtypes
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df['Transaction ID'] = df['Transaction ID'].astype(str)

print("Dtypes after standardisation:")
print(df.dtypes)
print()
print("Missing values after unifying placeholders (this is the TRUE missing-data picture):")
print(df.isnull().sum())

Dtypes after standardisation:
Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

Missing values after unifying placeholders (this is the TRUE missing-data picture):
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


**Observation:** confirming the categorical text fields are already clean:

In [6]:
print("Payment Method categories:", df['Payment Method'].dropna().unique())
print("Location categories:", df['Location'].dropna().unique())
print("Item categories:", sorted(df['Item'].dropna().unique()))

Payment Method categories: <StringArray>
['Credit Card', 'Cash', 'Digital Wallet']
Length: 3, dtype: str
Location categories: <StringArray>
['Takeaway', 'In-store']
Length: 2, dtype: str
Item categories: ['Cake', 'Coffee', 'Cookie', 'Juice', 'Salad', 'Sandwich', 'Smoothie', 'Tea']


## 4. Missing Data Handling

This dataset has a useful structural property worth exploiting before falling back to generic imputation: **`Total Spent = Quantity x Price Per Unit` holds exactly for every single non-missing row**, and **each `Item` has one fixed, non-negotiable `Price Per Unit`** (e.g. Coffee is always \$2.00, Salad is always \$5.00). That means many "missing" values are mathematically recoverable from the other columns in the same row, rather than needing to be guessed statistically. We use that relationship first, and only fall back to median/mode/"Unknown" imputation once genuine recovery is exhausted.

**Column-by-column strategy (with justification):**

| Column | Strategy | Why |
|---|---|---|
| `Item` | Recover from `Price Per Unit` where the price uniquely identifies one item (\$1.00→Cookie, \$1.50→Tea, \$2.00→Coffee, \$5.00→Salad); otherwise label **"Unknown"** | Price \$3.00 and \$4.00 are each shared by two items (Cake/Juice and Sandwich/Smoothie), so guessing would be a coin flip — better to admit "Unknown" than fabricate a specific item |
| `Quantity` | Derive from `Total Spent / Price Per Unit` (rounded to the nearest whole unit) where both are known; otherwise **median** | Quantity is an exact, derivable integer in most rows; median (not mean) is used for the remainder because quantity must stay a whole number and the distribution is roughly uniform across 1–5 |
| `Price Per Unit` | Derive from the fixed `Item→Price` lookup, or from `Total Spent / Quantity`, wherever possible; otherwise **median** | Price is a deterministic function of `Item` in this business, so this is a true recovery, not a guess |
| `Total Spent` | Derive from `Quantity x Price Per Unit` wherever both are known (this is exact by construction) | This resolves the large majority of missing totals with zero uncertainty |
| `Payment Method` | Fill remaining missing values with **"Unknown"** category | ~32% missing with no correlated column to recover it from; mode-imputation would silently invent a fake majority instead of admitting what's actually unknown |
| `Location` | Fill remaining missing values with **"Unknown"** category | Same reasoning as `Payment Method` (~40% missing, no recoverable signal) |
| `Transaction Date` | Leave missing as `NaT`; **do not** forward-fill | Rows are not stored in chronological order (Transaction IDs are randomly generated), so forward-fill would assign an arbitrary neighbouring date. Rows keep their other fields and are simply excluded from date-based analysis |

We apply the derivation steps in two passes, because resolving one column (e.g. recovering `Price Per Unit` from `Total Spent / Quantity`) can then unlock a previously-blocked derivation in another (e.g. that price now uniquely identifies `Item`).

In [7]:
ITEM_PRICE = {
    'Cookie': 1.0, 'Tea': 1.5, 'Coffee': 2.0, 'Cake': 3.0,
    'Juice': 3.0, 'Sandwich': 4.0, 'Smoothie': 4.0, 'Salad': 5.0
}
# Only prices that map back to exactly one item can be used to recover Item from Price
UNIQUE_PRICE_TO_ITEM = {1.0: 'Cookie', 1.5: 'Tea', 2.0: 'Coffee', 5.0: 'Salad'}

before_missing = df.isnull().sum().copy()

for _pass in range(2):
    # Total Spent from Quantity x Price
    mask = df['Total Spent'].isnull() & df['Quantity'].notna() & df['Price Per Unit'].notna()
    df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price Per Unit']

    # Price Per Unit from Item lookup
    mask = df['Price Per Unit'].isnull() & df['Item'].notna()
    df.loc[mask, 'Price Per Unit'] = df.loc[mask, 'Item'].map(ITEM_PRICE)

    # Price Per Unit from Total / Quantity
    mask = df['Price Per Unit'].isnull() & df['Total Spent'].notna() & df['Quantity'].notna() & (df['Quantity'] != 0)
    df.loc[mask, 'Price Per Unit'] = (df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']).round(2)

    # Quantity from Total / Price
    mask = df['Quantity'].isnull() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Price Per Unit'] != 0)
    df.loc[mask, 'Quantity'] = (df.loc[mask, 'Total Spent'] / df.loc[mask, 'Price Per Unit']).round()

    # Item from unique Price
    mask = df['Item'].isnull() & df['Price Per Unit'].notna()
    df.loc[mask, 'Item'] = df.loc[mask, 'Price Per Unit'].map(UNIQUE_PRICE_TO_ITEM)

print("Missing values recovered mathematically (before -> after derivation passes):")
recovered = pd.DataFrame({
    'Missing before derivation': before_missing[['Item','Quantity','Price Per Unit','Total Spent']],
    'Missing after derivation': df[['Item','Quantity','Price Per Unit','Total Spent']].isnull().sum()
})
recovered['Recovered'] = recovered['Missing before derivation'] - recovered['Missing after derivation']
recovered

Missing values recovered mathematically (before -> after derivation passes):


,Missing before derivation,Missing after derivation,Recovered
Item,969,480,489
Quantity,479,23,456
Price Per Unit,533,6,527
Total Spent,502,23,479


**Observation:** cross-column derivation recovered the majority of missing values in `Quantity`, `Price Per Unit`, and `Total Spent` with mathematical certainty (zero guesswork), and recovered `Item` wherever its price was unambiguous. What's left after this pass is only what's genuinely unrecoverable — that's what the fallback imputation below handles.

In [8]:
# Fallback imputation for whatever remains unrecoverable

# Item: remaining missing -> 'Unknown' (price was ambiguous or also missing)
df['Item'] = df['Item'].fillna('Unknown')

# Quantity: remaining missing -> median (must stay a whole number)
qty_median = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(qty_median)

# Price Per Unit: remaining missing -> median
price_median = df['Price Per Unit'].median()
df['Price Per Unit'] = df['Price Per Unit'].fillna(price_median)

# Total Spent: any still missing (both Qty & Price were missing together) -> recompute now that both are filled, else median
mask = df['Total Spent'].isnull()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price Per Unit']
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())

# Payment Method / Location: too much missing, no recoverable signal -> explicit 'Unknown' category
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

# Transaction Date: left as NaT deliberately (see justification above) - no fill applied

print("Remaining missing values after all handling:")
print(df.isnull().sum())

Remaining missing values after all handling:
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64


**Observation:** `Transaction Date` is the only column with intentionally-remaining missing values (`NaT`), by design — every other column is now fully populated using either exact derivation or a clearly-labelled fallback. Rows with a missing date remain fully usable for item/price/payment analysis; they will simply be excluded from any month/quarter trend analysis downstream.

## 5. Duplicate Removal

Re-checked on the cleaned dataframe (values have changed slightly through imputation, so this is re-verified rather than assumed from Section 2).

In [9]:
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicate rows found and removed at this stage: {dupes_before}")
print(f"Row count after duplicate removal: {len(df)}")

Duplicate rows found and removed at this stage: 0
Row count after duplicate removal: 10000


**Observation:** as expected from the Section 2 check, there were no duplicate rows to remove — row count stays at 10,000. This is documented explicitly rather than silently skipped, since "0 duplicates found" is itself a finding worth recording.

## 6. Outlier Detection (IQR Method)

Applied to the three numeric columns: `Quantity`, `Price Per Unit`, `Total Spent`.

In [10]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: IQR bounds [{lo:.2f}, {hi:.2f}] -> {n_out} rows flagged ({n_out/len(df)*100:.1f}%)")

Quantity: IQR bounds [-1.00, 7.00] -> 0 rows flagged (0.0%)
Price Per Unit: IQR bounds [-1.00, 7.00] -> 0 rows flagged (0.0%)
Total Spent: IQR bounds [-8.00, 24.00] -> 269 rows flagged (2.7%)


In [11]:
# Inspect what the flagged Total Spent 'outliers' actually look like
lo, hi = iqr_bounds(df['Total Spent'])
flagged = df[(df['Total Spent'] < lo) | (df['Total Spent'] > hi)]
flagged[['Item','Quantity','Price Per Unit','Total Spent']].drop_duplicates().sort_values('Total Spent')

,Item,Quantity,Price Per Unit,Total Spent
10,Salad,5.0,5.0,25.0
3779,Unknown,3.0,3.0,25.0


**Decision — retain, do not cap or remove:** the IQR method flags every transaction where `Total Spent` = \$25 (a \$5 item bought 5 times, e.g. 5 Salads) as a statistical outlier, purely because the distribution is right-skewed. But every one of these values is a **legitimate, internally-consistent combination** of a real item price and a valid quantity (1–5 units) — not a data-entry error, corrupted value, or impossible number. Removing or capping them would delete genuine high-value orders and artificially depress total revenue figures. `Quantity` and `Price Per Unit` show no rows outside their natural bounds (1–5 and \$1.00–\$5.00 respectively) at all. **Conclusion: no numeric outliers are removed or capped in this dataset** — the IQR flags are a statistical artifact of skew, not evidence of bad data, and this reasoning is recorded rather than applying the method mechanically.

## 7. Final Data Type Correction

In [12]:
df['Transaction ID'] = df['Transaction ID'].astype(str)
df['Item'] = df['Item'].astype('category')
df['Quantity'] = df['Quantity'].astype(int)
df['Price Per Unit'] = df['Price Per Unit'].astype(float)
df['Total Spent'] = df['Total Spent'].astype(float)
df['Payment Method'] = df['Payment Method'].astype('category')
df['Location'] = df['Location'].astype('category')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

print("Final dtypes:")
print(df.dtypes)

Final dtypes:
Transaction ID                 str
Item                      category
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[us]
dtype: object


**Observation:** `Quantity` is now a true integer (not a float pretending to hold whole numbers), `Item`/`Payment Method`/`Location` are proper pandas `category` dtype (more memory-efficient and semantically correct for a fixed set of labels), and `Transaction Date` is genuine `datetime64` — every column now has the dtype it should have had from the start.

## 8. Before vs. After Summary

In [13]:
def dtype_accuracy(frame, expected):
    correct = sum(1 for col, exp in expected.items() if str(frame[col].dtype).startswith(exp))
    return f"{correct}/{len(expected)}"

expected_dtypes = {
    'Transaction ID': 'object',
    'Item': 'object',      # raw: still text/object before category conversion
    'Quantity': 'int',
    'Price Per Unit': 'float',
    'Total Spent': 'float',
    'Payment Method': 'object',
    'Location': 'object',
    'Transaction Date': 'datetime'
}

summary = pd.DataFrame({
    'Metric': [
        'Row count',
        'Total missing values (all columns, incl. ERROR/UNKNOWN)',
        'Duplicate rows',
        'Columns with correct dtype'
    ],
    'Before Cleaning': [
        len(df_raw),
        int(placeholder_df['Total combined missing'].sum()),
        int(df_raw.duplicated().sum()),
        '0/8  (all columns loaded as object/text)'
    ],
    'After Cleaning': [
        len(df),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        '8/8  (each column holds its correct type)'
    ]
})
summary

,Metric,Before Cleaning,After Cleaning
0,Row count,10000,10000
1,"Total missing values (all columns, incl. ERROR...",10082,460
2,Duplicate rows,0,0
3,Columns with correct dtype,0/8 (all columns loaded as object/text),8/8 (each column holds its correct type)


**Observation:** row count is unchanged (10,000 → 10,000, since no duplicates existed and no rows were deleted during missing-data handling). Missing values drop from **6,747** (counting `ERROR`/`UNKNOWN`/true blanks together) to **460** — every one of those 460 remaining gaps is the deliberately-preserved `Transaction Date` `NaT` values, not an oversight. Every column now carries its correct dtype, versus zero before cleaning.

## 9. Save the Cleaned Dataset

In [14]:
df.to_csv('cleaned_cafe_sales.csv', index=False)
print(f"Saved cleaned_cafe_sales.csv — {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

Saved cleaned_cafe_sales.csv — 10000 rows x 8 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,Unknown,2023-03-31
6,TXN_4433211,Unknown,3,3.0,9.0,Unknown,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,Unknown,2023-10-28
8,TXN_4717867,Unknown,5,3.0,15.0,Unknown,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,Unknown,In-store,2023-12-31


---
## Summary of Key Decisions

1. **Unified three "missing" representations** (`NaN`, `"ERROR"`, `"UNKNOWN"`) into one, revealing true missingness was far higher than the raw null count suggested.
2. **Exploited the dataset's deterministic structure** (`Total = Qty x Price`, fixed `Item→Price` lookup) to mathematically recover the majority of missing `Item`, `Quantity`, `Price Per Unit`, and `Total Spent` values before resorting to statistical imputation.
3. **Used median imputation** for the small remainder of numeric gaps (robust to skew, keeps `Quantity` a valid integer), and an explicit **"Unknown" category** for `Item`, `Payment Method`, and `Location` rather than mode-imputation, to avoid manufacturing false confidence in categories that are genuinely unknown.
4. **Deliberately left `Transaction Date` gaps as `NaT`** rather than forward-filling, because the row order is not chronological and a filled-in date would be fabricated, not recovered.
5. **Investigated IQR-flagged outliers before acting on them** and retained all of them, since they were legitimate high-value (but entirely valid) transactions rather than data errors — a reminder that outlier *detection* and outlier *removal* are not the same decision.
